In [1]:
import sys
REPO_ROOT = "/mnt/custom-file-systems/s3/shared/parking_ml"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [2]:
# Config
S3_ROOT = "s3://smart-park-seattle/parking_v2/" #v2
YEAR = 2023

FEAT_TOPK = 16
TIN = 12
EPOCHS = 30
BATCH_SIZE = 4
LR = 1e-3
HIDDEN = 64

In [3]:
# Sanity Check
from parking_processing.utils.s3 import parse_s3_uri, s3_exists, s3_client

s3 = s3_client()
bucket, prefix = parse_s3_uri(S3_ROOT)

required = [
    f"{prefix}splits/year={YEAR}/splits.json",
    f"{prefix}graphs/year={YEAR}/nodes.csv",
    f"{prefix}graphs/year={YEAR}/topology_edges.csv",
]
for k in required:
    print(k, "=>", s3_exists(s3, bucket, k))

parking_v2/splits/year=2023/splits.json => True
parking_v2/graphs/year=2023/nodes.csv => True
parking_v2/graphs/year=2023/topology_edges.csv => True


In [4]:
import json, boto3

bucket = "smart-park-seattle"
splits_key = "parking_v2/splits/year=2023/splits.json"

s3 = boto3.client("s3")
splits = json.loads(s3.get_object(Bucket=bucket, Key=splits_key)["Body"].read())

def drop_missing(weeks):
    return [w for w in weeks if w != "2021-12-27"]

splits["train_weeks"] = drop_missing(splits["train_weeks"])
splits["val_weeks"]   = drop_missing(splits["val_weeks"])
splits["test_weeks"]  = drop_missing(splits["test_weeks"])

s3.put_object(Bucket=bucket, Key=splits_key, Body=json.dumps(splits, indent=2).encode("utf-8"))
print("done. train_weeks[0:3] =", splits["train_weeks"][:3])

done. train_weeks[0:3] = ['2022-12-26', '2023-01-02', '2023-01-09']


In [5]:
# Build MV-STGCN inputs (feature graph + panels)
from parking_processing.mv.build_inputs import run_build_inputs, BuildInputsCfg

run_build_inputs(BuildInputsCfg(
    s3_root=S3_ROOT,
    year=YEAR,
    feat_topk=FEAT_TOPK,
    sig_mode="dow_hour",
))

Signature (train weeks):   0%|          | 0/41 [00:00<?, ?it/s]


NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.

In [ ]:
# Training with y_15
from sagemaker.pytorch import PyTorch
from datetime import datetime
import sagemaker

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

est = PyTorch(
    entry_point="parking_processing/model/train_mvstgcn.py",
    source_dir=".",
    output_path="s3://smart-park-seattle/models_v2/", #v2
    role=role,
    framework_version="2.2.0",
    py_version="py310",
    instance_count=1,
    instance_type="ml.c4.2xlarge",
    hyperparameters={
        "s3_root": S3_ROOT,
        "year": YEAR,
        "target": "y_15",
        "tin": TIN,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "hidden": HIDDEN,
        "feat_topk": FEAT_TOPK,
        "use_topology": 1,
        "use_feature_graph": 1,
        "early_stop": 1,    ##############
        "patience": 6,      # early stop #
        "min_delta": 1e-4,  ##############
    },
)

job_name = f"mvstgcn-parking-v2-2022-y15-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
est.fit(job_name=job_name)

## Check CloudWatch Log for "mvstgcn-parking-v2-2022-y15-20260225-203551/algo-1-1772051799"

### 14 Epochs total but the best one was in 8

### Modify Config AND Sanity Check

In [7]:
YEAR = 2022

FEAT_TOPK = 16
TIN = 12
EPOCHS = 30
BATCH_SIZE = 4
LR = 7e-4
HIDDEN = 64

# This will be used for the y_30 training

In [8]:
# --- Sanity check for 2022 panels/labels  --- #

import os, io, json
import numpy as np
import pandas as pd
import boto3

def parse_s3_uri(uri: str):
    assert uri.startswith("s3://")
    x = uri[5:]
    bucket, *rest = x.split("/", 1)
    prefix = rest[0] if rest else ""
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    return bucket, prefix

def s3_get_text(s3, bucket, key) -> str:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")

def s3_download(s3, bucket, key, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(bucket, key, local_path)

def panel_paths_from_weeks(s3, bucket, prefix, year, weeks, local_dir="/tmp/panels_sanity"):
    paths = []
    missing = []
    for wk in weeks:
        key = f"{prefix}mvstgcn/year={year}/panels/week={wk}/panel.npz"
        local = os.path.join(local_dir, f"panel_{wk}.npz")
        try:
            s3_download(s3, bucket, key, local)
            paths.append(local)
        except Exception as e:
            missing.append((wk, str(e)))
    return paths, missing

def compute_stats(panel_files, target="y15"):
    # Aggregate basic stats across given panel.npz files
    tot_mask = 0
    tot_pos = 0
    tot_cnt = 0
    tot_minus1 = 0
    # Optional hourly slice
    hourly = np.zeros(24, dtype=np.float64)
    hourly_cnt = np.zeros(24, dtype=np.float64)

    for fp in panel_files:
        z = np.load(fp)
        X = z["X"]              # (T,N,F)
        y15, y30 = z["y15"], z["y30"]   # (T,N)
        m = z["mask"]
        m15, m30 = m, m   # (T,N)

        if target == "y15":
            y, m = y15, m15
        else:
            y, m = y30, m30

        # keep only observed points
        keep = (m > 0.5)
        yy = y[keep]

        # counts
        tot_mask += keep.sum()
        tot_minus1 += (yy < -0.5).sum()   # should be ~0 after masking if labeling is consistent

        # valid labels should be 0/1
        valid = (yy >= -0.5)
        yy2 = yy[valid]
        tot_cnt += yy2.size
        tot_pos += (yy2 > 0.5).sum()

        # Coverage: mask ratio against all slots
        # We track tot_mask only; full slots depends on T,N but we can estimate by summing keep + not keep
        tot_mask

        # Hour-of-day pos_rate slice (T=672, each hour has 4 bins)
        # This assumes panel time axis aligns to local week grid (your panelize does that).
        # Map each t to hour 0-23
        T = y.shape[0]
        hours = (np.arange(T) // 4) % 24   # 4*15min bins per hour
        # For each hour, accumulate positives / counts over all nodes
        for h in range(24):
            hh = (hours == h)
            k = keep[hh, :]
            if k.any():
                yyh = y[hh, :][k]
                # exclude -1 just in case
                yyh = yyh[yyh >= -0.5]
                if yyh.size:
                    hourly[h] += (yyh > 0.5).sum()
                    hourly_cnt[h] += yyh.size

    pos_rate = (tot_pos / max(tot_cnt, 1.0))
    minus1_rate = (tot_minus1 / max(tot_mask, 1.0))
    hourly_pos = np.divide(hourly, np.maximum(hourly_cnt, 1.0))

    return {
        "target": target,
        "label_cnt": int(tot_cnt),
        "pos_rate": float(pos_rate),
        "masked_minus1_rate": float(minus1_rate),  # should be ~0
        "hourly_pos_rate": hourly_pos,
        "hourly_cnt": hourly_cnt,
    }

# ---- RUN ----
s3 = boto3.client("s3")
bucket, prefix = parse_s3_uri(S3_ROOT)

splits_key = f"{prefix}splits/year={YEAR}/splits.json"
splits = json.loads(s3_get_text(s3, bucket, splits_key))

# Use val weeks (best quick sanity). If val empty, fall back to first few train weeks.
val_weeks = splits.get("val_weeks", [])
train_weeks = splits.get("train_weeks", [])
weeks = val_weeks if len(val_weeks) else train_weeks[:5]

print("Using weeks:", weeks[:10], f"(count={len(weeks)})")

panel_files, missing = panel_paths_from_weeks(s3, bucket, prefix, YEAR, weeks)

if missing:
    print("[WARN] Missing panels for weeks (showing up to 10):")
    for wk, err in missing[:10]:
        print(" -", wk, err)

print("Downloaded panels:", len(panel_files))
assert len(panel_files) > 0, "No panels downloaded. Check S3_ROOT/YEAR/panels path."

# y_15 sanity
st15 = compute_stats(panel_files, target="y_15")
print("\n=== SANITY: y_15 ===")
print("label_cnt:", st15["label_cnt"])
print("pos_rate:", st15["pos_rate"])
print("masked_minus1_rate (should be ~0):", st15["masked_minus1_rate"])
print("hourly_pos_rate (0~23):", np.round(st15["hourly_pos_rate"], 3))

# y_30 sanity (so you know what you'll train next)
st30 = compute_stats(panel_files, target="y_30")
print("\n=== SANITY: y_30 ===")
print("label_cnt:", st30["label_cnt"])
print("pos_rate:", st30["pos_rate"])
print("masked_minus1_rate (should be ~0):", st30["masked_minus1_rate"])
print("hourly_pos_rate (0~23):", np.round(st30["hourly_pos_rate"], 3))

# Basic red flags
def flag(st):
    flags = []
    if st["masked_minus1_rate"] > 1e-4:
        flags.append("masked_minus1_rate>0 (masking/labeling mismatch)")
    if st["pos_rate"] > 0.9:
        flags.append("pos_rate>0.9 (label too positive / has_space too lenient / aggregation wrong)")
    if st["pos_rate"] < 0.01:
        flags.append("pos_rate<0.01 (label too negative / has_space too strict)")
    return flags

print("\nRed flags y_15:", flag(st15) or "none")
print("Red flags y_30:", flag(st30) or "none")

Using weeks: ['2022-10-10', '2022-10-17', '2022-10-24', '2022-10-31'] (count=4)
Downloaded panels: 4

=== SANITY: y_15 ===
label_cnt: 1299043
pos_rate: 0.542249178818561
masked_minus1_rate (should be ~0): 0.0
hourly_pos_rate (0~23): [0.    0.    0.    0.    0.    0.    1.    0.    0.676 0.584 0.524 0.497
 0.494 0.515 0.552 0.562 0.551 0.521 0.431 0.549 0.55  0.68  0.    0.   ]

=== SANITY: y_30 ===
label_cnt: 1299043
pos_rate: 0.542249178818561
masked_minus1_rate (should be ~0): 0.0
hourly_pos_rate (0~23): [0.    0.    0.    0.    0.    0.    1.    0.    0.676 0.584 0.524 0.497
 0.494 0.515 0.552 0.562 0.551 0.521 0.431 0.549 0.55  0.68  0.    0.   ]

Red flags y_15: none
Red flags y_30: none


## y-30 and y-15 looks exactly the same

## We must know if this is because of faulty logic during preprocess OR characteristics of our dataset

In [21]:
z=np.load(panel_files[0])
print(np.mean(z["y15"]==z["y30"]))

# 98% similar

0.9824509086041824


In [22]:
z = np.load(panel_files[0])
m = z["mask"] > 0.5
print("eq_all :", np.mean(z["y15"] == z["y30"]))
print("eq_mask:", np.mean(z["y15"][m] == z["y30"][m]))

# some difference

eq_all : 0.9824509086041824
eq_mask: 0.944951947566815


In [16]:
# simlar positive rate?
z = np.load(panel_files[0])
m = z["mask"] > 0.5
print("pos_y15:", z["y15"][m].mean())
print("pos_y30:", z["y30"][m].mean())

# how often do they change?
diff = (z["y15"] != z["y30"]) & m
print("diff_rate:", diff.mean())

#mismatch in specific time period?
T = z["y15"].shape[0]
hours = (np.arange(T)//4) % 24
diff = (z["y15"] != z["y30"]) & (z["mask"]>0.5)
for h in range(24):
    kk = diff[hours==h]
    print(h, kk.mean() if kk.size else 0.0)

pos_y15: 0.5301574168691362
pos_y30: 0.5306297600928633
diff_rate: 0.017549091395817586
0 0.0
1 0.0
2 0.0
3 0.0
4 0.0
5 0.0
6 0.0
7 0.0
8 0.02166005291005291
9 0.030517762660619802
10 0.03517101284958428
11 0.037462207105064246
12 0.04083994708994709
13 0.0383834089191232
14 0.038052721088435375
15 0.038950302343159486
16 0.03510015117157974
17 0.035147392290249435
18 0.02343159486016629
19 0.034958427815570674
20 0.004747732426303855
21 0.006755479969765684
22 0.0
23 0.0


In [23]:
z = np.load(panel_files[0])
m = z["mask"] > 0.5
print("y15_missing_in_mask:", np.mean(z["y15"][m] < 0))
print("y30_missing_in_mask:", np.mean(z["y30"][m] < 0))

print("y15!=y30 (mask only):", np.mean(z["y15"][m] != z["y30"][m]))

y15_missing_in_mask: 0.0
y30_missing_in_mask: 0.0
y15!=y30 (mask only): 0.055048052433185043


In [ ]:
# Training with y_30
from sagemaker.pytorch import PyTorch
from datetime import datetime
import sagemaker

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

est = PyTorch(
    entry_point="parking_processing/model/train_mvstgcn.py",
    source_dir=".",
    output_path="s3://smart-park-seattle/models_v2/", #v2
    role=role,
    framework_version="2.2.0",
    py_version="py310",
    instance_count=1,
    instance_type="ml.c4.2xlarge",
    hyperparameters={
        "s3_root": S3_ROOT,
        "year": YEAR,
        "target": "y_30",
        "tin": TIN,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "hidden": HIDDEN,
        "feat_topk": FEAT_TOPK,
        "use_topology": 1,
        "use_feature_graph": 1,
        "early_stop": 1,    ##############
        "patience": 3,      # early stop #
        "min_delta": 1e-4,  ##############
    },
)

job_name = f"mvstgcn-parking-v2-2022-y30-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
est.fit(job_name=job_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-26 09:02:21 Starting - Starting the training job...
2026-02-26 09:02:52 Starting - Preparing the instances for training...
2026-02-26 09:03:14 Downloading - Downloading input data...
2026-02-26 09:03:34 Downloading - Downloadin

## Training for y-15 and y-30 Done